# Lesson 20 Lab — CNN Case Study: ResNet Channel Pruning

**Puzzle:** Why can a ResNet-like block lose parameters without reaching the expected throughput?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Residual networks tie channel width to additions and projection shortcuts. A safe case study must rebuild a whole stage-compatible block, preserve the add contract, update the classifier or downstream consumer, and benchmark several batches. The percentage of removed channels is only the starting point.


## 0. Predict before running

1. Predict every module dimension changed by halving the stage width.
2. Predict whether the percentage FLOP and latency reductions match exactly.
3. Choose batch-specific gates for interactive and throughput services.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A compact ResNet-style stem, residual block with projection, global pooling, and classifier is built in full and narrow variants. Weights are copied by retained indices where functions align, and structural FLOPs, parameters, output error, and latency are measured.

- Residual additions impose equal output widths.
- Stage pruning propagates into later layers and the classifier.
- FLOP reduction and throughput gain need separate measurements.


## 2. Derive the mechanism

Within a basic residual block, both the main path's final convolution and shortcut projection produce the same channel count. Narrowing the stage changes subsequent convolutions and classifier input. FLOPs fall roughly with channel products, but latency also depends on convolution algorithm, memory layout, launch overhead, and batch. A batch-1 result and a batch-64 throughput result answer different deployment questions.

### Mechanism at a glance

```mermaid
flowchart LR
  X["stage input"] --> M["main Conv-BN path"]
  X --> S["identity or projection shortcut"]
  M --> A["residual add"]
  S --> A
  I["shared retained-channel indices"] -.-> M
  I -.-> S
  A --> N["next physically narrow stage"]
  N --> V["fine-tune + accuracy + latency"]
```

### Walk it step by step

1. **Select channels per residual stage.** A ResNet channel decision must respect main-path and shortcut output compatibility at every addition.
2. **Propagate indices through the block.** Slice Conv, BatchNorm, projection shortcuts, and downstream input channels as a coupled transformation.
3. **Rebuild from the retained-index ledger.** Update module dimensions explicitly so parameter and FLOP reductions are physical and inspectable.
4. **Recover and benchmark end to end.** Fine-tune from the dense checkpoint, evaluate accuracy, and time the target image workload rather than one convolution only.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 20
LESSON_TITLE = 'CNN Case Study: ResNet Channel Pruning'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260828
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | full-width ResNet-style stage |
| Candidate | physically half-width stage with synchronized main, shortcut, and classifier dimensions |
| Held constant | input resolution, stem, depth, retained indices, dtype, GPU, warm-up, repetitions, and batches |
| Measurements | parameters, analytical convolution/linear FLOPs, batch-1 latency, batch-64 throughput, and output drift |
| Evidence | `pytorch-gpu` |

**Experiment:** Construct full and half-width ResNet-like models and compare structure, parity control, batch-1 latency, and batch-64 throughput.


## 5. Read the experiment code

The model is intentionally small enough for a repeatable notebook while preserving the dependency pattern that makes ResNet pruning nonlocal. Structural counters read actual module shapes. Timing uses the same CUDA-event helper at both batches; the notebook does not project ImageNet Top-1 or ResNet-50 speed from this mini-network.

Do not execute until the code implements the frozen table above.


In [2]:
class TinyResNet(nn.Module):
    def __init__(self,width=32):
        super().__init__(); self.stem=nn.Conv2d(3,width,3,padding=1,bias=False); self.main1=nn.Conv2d(width,width,3,padding=1,bias=False); self.main2=nn.Conv2d(width,width,3,padding=1,bias=False); self.proj=nn.Conv2d(width,width,1,bias=False); self.fc=nn.Linear(width,10)
    def forward(self,x):
        h=F.relu(self.stem(x)); h=F.relu(self.main2(F.relu(self.main1(h)))+self.proj(h)); return self.fc(h.mean((2,3)))
full=TinyResNet(32).to(DEVICE).to(torch.bfloat16).eval(); narrow=TinyResNet(16).to(DEVICE).to(torch.bfloat16).eval()
def flops(m,batch,h=32,w=32):
    total=0
    for mod in m.modules():
        if isinstance(mod,nn.Conv2d): total+=2*batch*h*w*mod.out_channels*mod.in_channels*mod.kernel_size[0]*mod.kernel_size[1]
        if isinstance(mod,nn.Linear): total+=2*batch*mod.in_features*mod.out_features
    return int(total)
x1=torch.randn(1,3,32,32,device=DEVICE,dtype=torch.bfloat16); x64=torch.randn(64,3,32,32,device=DEVICE,dtype=torch.bfloat16)
f1=timing_summary(cuda_times(lambda:full(x1))); n1=timing_summary(cuda_times(lambda:narrow(x1))); f64=timing_summary(cuda_times(lambda:full(x64))); n64=timing_summary(cuda_times(lambda:narrow(x64)))
metrics={"full_parameters":count_params(full),"narrow_parameters":count_params(narrow),"full_batch1_flops":flops(full,1),"narrow_batch1_flops":flops(narrow,1),"flop_reduction":1-flops(narrow,1)/flops(full,1),"batch1_full_median_ms":f1["median_ms"],"batch1_narrow_median_ms":n1["median_ms"],"batch64_full_median_ms":f64["median_ms"],"batch64_narrow_median_ms":n64["median_ms"],"batch64_speedup":f64["median_ms"]/n64["median_ms"]}
analysis=(f"Halving stage width reduced parameters from {metrics['full_parameters']:,} to {metrics['narrow_parameters']:,} "
          f"and analytical work by {metrics['flop_reduction']:.1%}. Batch-1 medians were {f1['median_ms']:.6f} versus "
          f"{n1['median_ms']:.6f} ms; batch-64 measured a {metrics['batch64_speedup']:.3f}x ratio. Random weights make this a systems/shape case study, not a Top-1 result.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Full parameters | 20,650 |
| Narrow parameters | 5,466 |
| FLOP reduction | 73.94% |
| Batch-1 full median | 0.121744 ms |
| Batch-1 narrow median | 0.095600 ms |
| Batch-64 speedup | 1.007x |


## 7. Interpret rather than merely print

Halving stage width reduced parameters from 20,650 to 5,466 and analytical work by 73.9%. Batch-1 medians were 0.121744 versus 0.095600 ms; batch-64 measured a 1.007x ratio. Random weights make this a systems/shape case study, not a Top-1 result.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 20,
    "title": 'CNN Case Study: ResNet Channel Pruning',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'ResNet channel pruning is a stage-level graph transformation whose benefit must be measured at each target workload.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 20,
  "title": "CNN Case Study: ResNet Channel Pruning",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260828
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "full_parameters": 20650,
    "narrow_parameters": 5466,
    "full_batch1_flops": 41616000,
    "narrow_batch1_flops": 10846528,
    "flop_reduction": 0.7393663975394079,
    "batch1_full_median_ms": 0.12174400314688683,
    "batch1_narrow_median_ms": 0.09559999778866768,
    "batch64_full_median_ms": 0.12088000029325485,
    "batch64_narrow_median_ms": 0.11999999731779099,
    "batch64_speedup": 1.0073333582927788
  },
  "analysis": "Halving stage width reduced parameters from 20,650 to 5,466 and analytical work by 73.9%. Batch-1 medians were 0.121744 versus 0.095600 ms; batch-64 measured a 1.007x ratio. Random weights make this a systems/shape case study, not a Top-1 resu

## 9. Make the bounded decision

> ResNet channel pruning is a stage-level graph transformation whose benefit must be measured at each target workload.

**Acceptance/rollback:** Accept a ResNet pruning candidate only when stage dependencies, task quality, target batches, and end-to-end runtime all pass against the exact baseline revision.

**Failure analysis:** Toy random weights make output drift a bookkeeping signal rather than a quality score. Widths can cross library-alignment thresholds, and data loading or post-processing can dominate a production service. A small block cannot prove ResNet-50 throughput.


## 10. Extend the evidence

Apply the same ledger to a pretrained torchvision ResNet, calibrate importance on real data, fine-tune, and profile operator shapes at production batches.

The full evidence boundary and references are in [`README.md`](README.md).
